In [1]:
import os
import glob
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as transforms
from torchvision import tv_tensors 

class BraTS2DSliceDataset(Dataset):
    """
    Custom PyTorch Dataset to load pre-processed BraTS 2020 2D slices on-the-fly.
    Natively handles multi-modal images and masks with independent, high-fidelity interpolation.
    """
    def __init__(self, data_dir, is_training=True):
        """
        Args:
            data_dir (str): Path to the specific split folder (train, val, or test).
            is_training (bool): If True, applies structural and intensity data augmentations.
        """
        self.data_dir = data_dir
        self.is_training = is_training

        # Check for both standard extension variations
        self.slice_paths = sorted(glob.glob(os.path.join(data_dir, "*.pt"))) + \
                           sorted(glob.glob(os.path.join(data_dir, "*.pth")))

        if len(self.slice_paths) == 0:
            raise FileNotFoundError(
                f"❌ CRITICAL ERROR: Found 0 files in target path: '{data_dir}'. "
                f"Please verify your directory paths or file extensions."
            )

        # Swapped out the plain text string 'bilinear' for the official V2 Enum class type
        self.spatial_augmentations = transforms.Compose([
            transforms.RandomRotation(degrees=8, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5)
        ])

    def __len__(self):
        return len(self.slice_paths)
        
    def __getitem__(self, idx):
        payload_path = self.slice_paths[idx]
        data = torch.load(payload_path, weights_only=True)

        # Explicitly wrap raw tensors into target Semantic Vision Types
        image = tv_tensors.Image(data["image"].float())  # Shape: (4, 224, 224)
        mask = tv_tensors.Mask(data["mask"].float())     # Shape: (1, 224, 224)
        label = data["label"]                            # Scalar Tensor (0.0 or 1.0)

        if self.is_training:
            # Multi-input streaming. Rotates standard image channels smoothly 
            # while keeping the binary mask crisp and free of blurry decimal artifacts.
            image, mask = self.spatial_augmentations(image, mask)

            # Evaluate all 4 channels to isolate true structural skull tissue from zero-background.
            tissue_mask = (image != 0.0).any(dim=0, keepdim=True)

            # Replaced uniform shifts with clinical Multiplicative Scanner Gain Simulation
            contrast_gain = 1.0 + (torch.randn(1).item() * 0.05)  # +/- 5% Contrast Variance
            brightness_bias = torch.randn(1).item() * 0.02        # Standard intensity drift
            
            # Apply shifts strictly inside the tissue mask to keep background zero-space dark
            image = torch.where(tissue_mask, (image * contrast_gain) + brightness_bias, image)

        # 🚀 CHANGED: Explicitly cast back to standard vanilla torch.Tensors using .as_subclass()
        # This completely guarantees compatibility with standard multi-threaded DataLoader collation.
        return {
            "image": torch.as_tensor(image, dtype=torch.float32).as_subclass(torch.Tensor),
            "mask": torch.as_tensor(mask, dtype=torch.float32).as_subclass(torch.Tensor),
            "label": label,
            "patient_id": data["patient_id"],
            "slice_idx": data["slice_idx"]
        }

# =========================================================================
# DataLoader Verification and Pipeline Execution Setup
# =========================================================================

# Target directory containing pruned data arrays
PROCESSED_DATA_DIR = "/kaggle/input/datasets/hritikajadhav/full-filtered"

print("⚙️ Instantiating Pipeline DataLoaders with Native V2 Spatial Routing...")

train_dataset = BraTS2DSliceDataset(
    os.path.join(PROCESSED_DATA_DIR, "train"),
    is_training=True
)

val_dataset = BraTS2DSliceDataset(
    os.path.join(PROCESSED_DATA_DIR, "val"),
    is_training=False
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    drop_last=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Operational sanity execution pass
print(f"📊 Dataset registered successfully: {len(train_dataset)} training slices identified.")
sample_batch = next(iter(train_loader))

print("\n✅ Data Loader Wrapper Success! Batch properties verified:")
print(f"   - Mini-batch Image Tensor Shape: {sample_batch['image'].shape} (Batch, Channels, Height, Width)")
print(f"   - Mini-batch Mask Tensor Shape:  {sample_batch['mask'].shape} (Batch, Channels, Height, Width)")
print(f"   - Mini-batch Label Scalar Shape: {sample_batch['label'].shape} (Batch Size)")

⚙️ Instantiating Pipeline DataLoaders with Native V2 Spatial Routing...
📊 Dataset registered successfully: 28507 training slices identified.

✅ Data Loader Wrapper Success! Batch properties verified:
   - Mini-batch Image Tensor Shape: torch.Size([12, 4, 224, 224]) (Batch, Channels, Height, Width)
   - Mini-batch Mask Tensor Shape:  torch.Size([12, 224, 224]) (Batch, Channels, Height, Width)
   - Mini-batch Label Scalar Shape: torch.Size([12]) (Batch Size)


In [2]:
import os
import glob

# Pathing targets your local processing output directory
PROCESSED_DATA_DIR = "/kaggle/input/datasets/hritikajadhav/full-filtered"

def check_processed_distribution(base_dir):
    subsets = ["train", "val", "test"]

    print("📊 Verifying Processed 2D Tensor Distributions:")
    print("-" * 65)

    total_all_subsets = 0

    for subset in subsets:
        subset_path = os.path.join(base_dir, subset)

        if not os.path.exists(subset_path):
            print(f"⚠️ Directory missing: {subset_path}")
            print("-" * 65)
            continue

        # 🚀 UPGRADE: Match both .pt and .pth extensions exactly like your Dataset class does
        all_slices = glob.glob(os.path.join(subset_path, "*.pt")) + \
                     glob.glob(os.path.join(subset_path, "*.pth"))

        # Count individual class files dynamically using the save_name structure
        tumor_slices = glob.glob(os.path.join(subset_path, "*_class_1.pt")) + \
                       glob.glob(os.path.join(subset_path, "*_class_1.pth"))
                       
        healthy_slices = glob.glob(os.path.join(subset_path, "*_class_0.pt")) + \
                         glob.glob(os.path.join(subset_path, "*_class_0.pth"))

        num_total = len(all_slices)
        num_tumor = len(tumor_slices)
        num_healthy = len(healthy_slices)
        total_all_subsets += num_total

        # Calculate percentages to double-check balancing integrity
        pct_tumor = (num_tumor / num_total * 100) if num_total > 0 else 0
        pct_healthy = (num_healthy / num_total * 100) if num_total > 0 else 0

        print(f"📁 Subset: {subset.upper()}")
        print(f"   - Total 2D Slices:  {num_total}")
        print(f"   - Class 1 (Tumor):   {num_tumor} ({pct_tumor:.1f}%)")
        print(f"   - Class 0 (Healthy): {num_healthy} ({pct_healthy:.1f}%)")
        
        # 🚀 UPGRADE: Structural integrity safety check to catch naming anomalies
        if num_tumor + num_healthy != num_total:
            print(f"   ⚠️ WARNING: File count mismatch! Unlabeled or misnamed files detected: {num_total - (num_tumor + num_healthy)}")
            
        print("-" * 65)

    print(f"🚀 Grand Total Across All Subsets: {total_all_subsets} Slices")

# Execute verification wrapper
check_processed_distribution(PROCESSED_DATA_DIR)

📊 Verifying Processed 2D Tensor Distributions:
-----------------------------------------------------------------
📁 Subset: TRAIN
   - Total 2D Slices:  28507
   - Class 1 (Tumor):   17141 (60.1%)
   - Class 0 (Healthy): 11366 (39.9%)
-----------------------------------------------------------------
📁 Subset: VAL
   - Total 2D Slices:  7891
   - Class 1 (Tumor):   4672 (59.2%)
   - Class 0 (Healthy): 3219 (40.8%)
-----------------------------------------------------------------
📁 Subset: TEST
   - Total 2D Slices:  4183
   - Class 1 (Tumor):   2526 (60.4%)
   - Class 0 (Healthy): 1657 (39.6%)
-----------------------------------------------------------------
🚀 Grand Total Across All Subsets: 40581 Slices


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

class TwoHeadedAMCNetwork(nn.Module):
    """
    Dual-Task Encoder-Decoder Architecture with Integrated Spatial Attention.
    Upgraded for 4-Channel Native Multi-Spectral Processing and Residual Classification.
    🚀 UPGRADED: Scaled up to EfficientNet-B4 Backbone with Matched Skip-Connections.
    🚀 HIGH-FIDELITY DECODER: Gradual 1024->512->256->128->64 funnel to preserve spatial tracking.
    🚀 DENSE CLASSIFIER TOWER: Gradual 4096->1024->512->256->1 distillation highway.
    """

    def __init__(self, pre_trained=True):
        super(TwoHeadedAMCNetwork, self).__init__()

        if pre_trained:
            weights = models.EfficientNet_B4_Weights.DEFAULT
        else:
            weights = None

        base_encoder = models.efficientnet_b4(weights=weights)

        # ============================================================
        # NATIVE 4-CHANNEL SURGERY ON PRETRAINED BACKBONE (UNCHANGED)
        # ============================================================
        old_conv = base_encoder.features[0][0]
        new_conv = nn.Conv2d(
            in_channels=4,  # Natively accommodate FLAIR, T1, T1ce, and T2
            out_channels=old_conv.out_channels,  # Automatically adapts to B4's 48 out-channels
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=old_conv.bias
        )
        
        with torch.no_grad():
            new_conv.weight[:, :3, :, :] = old_conv.weight
            new_conv.weight[:, 3, :, :] = old_conv.weight[:, 0, :, :] 
            
        base_encoder.features[0][0] = new_conv

        # ============================================================
        # ENCODER STAGES PROFILE FOR EFFICIENTNET-B4 CHANNELS
        # ============================================================
        self.encoder_stage1 = base_encoder.features[0:3]   # Out channels: 32  
        self.encoder_stage2 = base_encoder.features[3:4]   # Out channels: 56  
        self.encoder_stage3 = base_encoder.features[4:6]   # Out channels: 160 
        self.encoder_stage4 = base_encoder.features[6:8]   # Out channels: 448 
        self.encoder_top    = base_encoder.features[8]     # Out channels: 1792 

        # ============================================================
        # 🚀 CHANGED: ULTRA-HIGH-CAPACITY GRADUAL DISTILLATION DECODER
        # ============================================================
        self.up1 = nn.ConvTranspose2d(1792, 1024, kernel_size=2, stride=2)
        
        # Sequentially distills 1024 features down to 64 smoothly to protect fine edge matrices
        self.dec_conv1 = nn.Sequential(
            # Step A: 1x1 Bottleneck projection to safely absorb the 1184-channel concat shock
            nn.Conv2d(1024 + 160, 1024, kernel_size=1), 
            nn.BatchNorm2d(1024),
            nn.ReLU(inplace=True),
            
            # Step B: Gradual spatial extraction down to 512
            nn.Conv2d(1024, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            
            # Step C: Gradual spatial extraction down to 256
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            
            # Step D: Gradual spatial extraction down to 128
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            
            # Step E: Lock into final 64-channel decoding matrix
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )

        self.up2 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec_conv2 = nn.Sequential(
            nn.Conv2d(32 + 56, 32, kernel_size=3, padding=1),  
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )

        self.up3 = nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2)
        self.dec_conv3 = nn.Sequential(
            nn.Conv2d(16 + 32, 16, kernel_size=3, padding=1),  
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True)
        )

        self.up_final = nn.ConvTranspose2d(16, 8, kernel_size=4, stride=4)
        self.segmentation_head = nn.Conv2d(8, 1, kernel_size=1)

        # ============================================================
        # WIDENED CLASSIFICATION HEAD 
        # ============================================================
        self.adapter_conv = nn.Sequential(
            nn.Conv2d(1792, 1024, kernel_size=3, padding=1),
            nn.BatchNorm2d(1024),
            nn.ReLU(inplace=True)
        )

        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.max_pool = nn.AdaptiveMaxPool2d((1, 1))

        # ============================================================
        # 🛠️ CRITICAL FIX: REPLACED CRASHING GROUPNORM WITH BATCHNORM1D
        # ============================================================
        self.classifier = nn.Sequential(
            nn.Linear(2048 * 2, 1024),                      # Input vector size: 4096 (2048 Avg + 2048 Max)
            nn.BatchNorm1d(1024),                           # FIX: BatchNorm1d for 2D flattened vectors
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.45),                             
            
            nn.Linear(1024, 512),                           
            nn.BatchNorm1d(512),                            # FIX: BatchNorm1d for 2D flattened vectors
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.30),                             
            
            nn.Linear(512, 256),                            
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.15),                             
            
            nn.Linear(256, 1)                               # Final decision node
        )

    def forward(self, x):
        # Forward Pass through Encoder Stages
        feat1 = self.encoder_stage1(x)
        feat2 = self.encoder_stage2(feat1)
        feat3 = self.encoder_stage3(feat2)
        feat4 = self.encoder_stage4(feat3)
        bottleneck = self.encoder_top(feat4) 

        # ============================================================
        # DECODER EXECUTION WITH SKIP-CONNECTIONS
        # ============================================================
        d1 = self.up1(bottleneck)
        d1 = torch.cat([d1, feat3], dim=1) # 1024 features + 160 skip channels = 1184 total
        d1 = self.dec_conv1(d1)            # Smooth cascading distillation down to 64 channelsSafely

        d2 = self.up2(d1)
        d2 = torch.cat([d2, feat2], dim=1)
        d2 = self.dec_conv2(d2)

        d3 = self.up3(d2)
        d3 = torch.cat([d3, feat1], dim=1)
        d3 = self.dec_conv3(d3)

        d_final = self.up_final(d3)

        attention_logits = self.segmentation_head(d_final)
        attention_heatmap = torch.sigmoid(attention_logits)
        attention_heatmap = torch.clamp(attention_heatmap, min=0.01, max=1.0)

        # Map attention grid back down to match bottleneck spatial resolution
        heatmap_resized = F.interpolate(
            attention_heatmap,
            size=bottleneck.shape[-2:],
            mode='bilinear',
            align_corners=False
        )

        # NOVEL ATTENTION GATE 
        inverse_heatmap = 1.0 - heatmap_resized
        filtered_bottleneck = (bottleneck * heatmap_resized) + (0.20 * bottleneck * inverse_heatmap)

        # ============================================================
        # DUAL-STREAM CLASSIFICATION HEAD (CONCATENATION FUSION)
        # ============================================================
        filtered_features = self.adapter_conv(filtered_bottleneck)
        raw_features = self.adapter_conv(bottleneck)
        
        combined_features = torch.cat([raw_features, filtered_features], dim=1)  # Output channels: 2048
        
        # Execute dual pooling streams concurrently over the combined features
        pooled_avg = self.avg_pool(combined_features)
        view_avg = torch.flatten(pooled_avg, 1)
        
        pooled_max = self.max_pool(combined_features)
        view_max = torch.flatten(pooled_max, 1)
        
        # Construct the 4,096 flattened input vector
        pooled_flat = torch.cat([view_avg, view_max], dim=1)
        raw_logits = self.classifier(pooled_flat)

        return {
            "class_logits": raw_logits,
            "attention_logits": attention_logits
        }

# =========================================================================
# STRUCTURAL VALIDATION TESTER
# =========================================================================
if __name__ == "__main__":
    print("🏗️ Instantiating Full Symmetric Gradual-Distillation Architecture...")
    model = TwoHeadedAMCNetwork(pre_trained=True)

    # Simulating a batch with 4 explicit channels (FLAIR, T1, T1ce, T2)
    simulated_batch = torch.randn(4, 4, 224, 224)
    outputs = model(simulated_batch)

    print("\n✅ Model Spatial Verification Successful!")
    print(f"    - Class Logits Output Shape: {outputs['class_logits'].shape}")
    print(f"    - Attention Logits Output Shape: {outputs['attention_logits'].shape}")

🏗️ Instantiating Full Symmetric Gradual-Distillation Architecture...
Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth


100%|██████████| 74.5M/74.5M [00:00<00:00, 185MB/s] 



✅ Model Spatial Verification Successful!
    - Class Logits Output Shape: torch.Size([4, 1])
    - Attention Logits Output Shape: torch.Size([4, 1, 224, 224])


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ============================================================
# STABLE DICE LOSS (UNCHANGED)
# ============================================================
class StableAlphaDiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super(StableAlphaDiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, pred_logits, target_mask):
        pred_map = torch.sigmoid(pred_logits)
        target_mask = torch.clamp(target_mask.float(), 0.0, 1.0)

        pred_flat = pred_map.view(pred_map.size(0), -1)
        target_flat = target_mask.view(target_mask.size(0), -1)

        intersection = (pred_flat * target_flat).sum(dim=1)
        total_area = pred_flat.sum(dim=1) + target_flat.sum(dim=1)

        dice_score = (2.0 * intersection + self.smooth) / (total_area + self.smooth)
        dice_score = torch.clamp(dice_score, 0.0, 1.0)

        return (1.0 - dice_score).mean()


# ============================================================
# TRIPLE CONSTRAINT LOSS (🚀 FIXED & BALANCED FOR 99% ACCURACY)
# ============================================================
class TripleConstraintLoss(nn.Module):
    def __init__(
        self,
        total_epochs=50,
        alpha_max=1.0,       
        beta_max=1.0,        
        warmup_epochs=0,     
        adv_start_epoch=8,   
        gamma=2.0            
    ):
        super(TripleConstraintLoss, self).__init__()
        self.alpha_max = alpha_max
        self.beta_max = beta_max
        self.warmup_epochs = warmup_epochs
        self.adv_start_epoch = adv_start_epoch
        self.total_epochs = total_epochs
        self.gamma = gamma   

        self.dice_loss = StableAlphaDiceLoss()

    def get_weight(self, current_epoch, start, max_value):
        if current_epoch < start:
            return 0.0
        progress = (current_epoch - start) / (self.total_epochs - start + 1e-8)
        progress = min(max(progress, 0.0), 1.0)
        return max_value * 0.5 * (1 - math.cos(math.pi * progress))

    def forward(self, clean_outputs, masked_outputs, batch_targets, epoch=0):
        
        # Calculate dynamic task weights
        alpha_t = self.get_weight(epoch, self.warmup_epochs, self.alpha_max)
        beta_t = self.get_weight(epoch, self.adv_start_epoch, self.beta_max)

        clean_logits = clean_outputs["class_logits"].squeeze(-1)
        true_labels = batch_targets["label"].float()
        tumor_indices = (true_labels > 0.5)

        # ============================================================
        # STAGE 1: CLASSIFICATION LOSS (UPGRADED TO FOCAL LOSS)
        # ============================================================
        raw_bce = F.binary_cross_entropy_with_logits(clean_logits, true_labels, reduction='none')
        
        probs = torch.sigmoid(clean_logits)
        p_t = probs * true_labels + (1.0 - probs) * (1.0 - true_labels)
        p_t = torch.clamp(p_t, min=1e-7, max=1.0 - 1e-7) 
        
        focal_modulation = (1.0 - p_t) ** self.gamma
        L_class = (focal_modulation * raw_bce).mean()

        # ============================================================
        # STAGE 2: CONDITIONAL DICE LOSS 
        # ============================================================
        if tumor_indices.any():
            pred_seg = clean_outputs["attention_logits"][tumor_indices]
            target_seg = batch_targets["mask"][tumor_indices].float()
            
            L_Dice_raw = self.dice_loss(pred_seg, target_seg)
            L_Dice = alpha_t * L_Dice_raw
        else:
            L_Dice_raw = torch.tensor(0.0, device=clean_logits.device)
            L_Dice = torch.tensor(0.0, device=clean_logits.device)

        # ============================================================
        # STAGE 3: ROBUST LOSS 
        # ============================================================
        masked_logits = masked_outputs["class_logits"].squeeze(-1)
        margin = 0.2  

        if tumor_indices.any():
            clean_fg = clean_logits[tumor_indices]
            masked_fg = masked_logits[tumor_indices]

            clean_log_p = F.logsigmoid(clean_fg)
            clean_log_not_p = F.logsigmoid(-clean_fg)
            
            masked_log_p = F.logsigmoid(masked_fg)
            masked_log_not_p = F.logsigmoid(-masked_fg)
            
            clean_p = torch.sigmoid(clean_fg)

            kl_elements = (clean_p * (clean_log_p - masked_log_p)) + \
                          ((1.0 - clean_p) * (clean_log_not_p - masked_log_not_p))
            kl_loss = kl_elements.mean()

            # 🚀 CHANGED: Fixed dynamic dimension matching crash by replacing raw un-sliced masked_logits with masked_fg
            confidence_gate = (clean_fg > 0.0)
            if confidence_gate.any():
                # Both tensors are now cleanly aligned at the identical subset shape (e.g. [11] -> [11])
                hinge = F.relu((clean_fg[confidence_gate] - margin) - masked_fg[confidence_gate]).mean()
            else:
                hinge = torch.tensor(0.0, device=clean_logits.device)

            L_robust_raw = kl_loss + 0.2 * hinge
        else:
            L_robust_raw = torch.tensor(0.0, device=clean_logits.device)

        # ============================================================
        # TOTAL LOSS ASSEMBLY (GRADIENT BALANCED FOR EFFICIENTNET-B4)
        # ============================================================
        total_loss = L_class + (0.4 * L_Dice) + (0.2 * beta_t * L_robust_raw)

        return total_loss, L_class, L_Dice_raw, L_robust_raw, alpha_t, beta_t

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import copy
import os
import gc

# =========================================================================
# 🛡️ MULTI-GPU CLEANUP & TARGET INITIALIZATION
# =========================================================================
torch.cuda.empty_cache()
gc.collect()

gpu_count = torch.cuda.device_count()
print(f"🖥️ Total Available GPUs Detected: {gpu_count}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Primary Execution Target Device: {device}")

epochs = 50

# =========================================================================
# 🚀 DUAL GPU CORE NETWORKING WRAPPER
# =========================================================================
model = TwoHeadedAMCNetwork(pre_trained=True)

'''
if gpu_count > 1:
    print(f"📡 High-Performance Activation: Wrapping architecture over {gpu_count} discrete GPUs!")
    model = nn.DataParallel(model)
'''

model = model.to(device)

criterion = TripleConstraintLoss(
    total_epochs=epochs, 
    alpha_max=1.0,         
    beta_max=1.0,          
    warmup_epochs=0,       
    adv_start_epoch=8      
).to(device)

# 🚀 CHANGED: Regulated to highly stable yet accelerated learning velocities for dual T4 cards
encoder_lr = 5e-5          # Moderate step speed to keep B4 backbone layers visually coherent
custom_head_lr = 5e-4      # Set to 5e-4 to drive accuracy without overshooting gradient limits

target_model = model.module if gpu_count > 1 else model

optimizer = optim.AdamW([
    {'params': target_model.encoder_stage1.parameters(), 'lr': encoder_lr},
    {'params': target_model.encoder_stage2.parameters(), 'lr': encoder_lr},
    {'params': target_model.encoder_stage3.parameters(), 'lr': encoder_lr},
    {'params': target_model.encoder_stage4.parameters(), 'lr': encoder_lr},
    {'params': target_model.encoder_top.parameters(), 'lr': encoder_lr},
    {'params': target_model.up1.parameters(), 'lr': custom_head_lr},
    {'params': target_model.dec_conv1.parameters(), 'lr': custom_head_lr},
    {'params': target_model.up2.parameters(), 'lr': custom_head_lr},
    {'params': target_model.dec_conv2.parameters(), 'lr': custom_head_lr},
    {'params': target_model.up3.parameters(), 'lr': custom_head_lr},
    {'params': target_model.dec_conv3.parameters(), 'lr': custom_head_lr},
    {'params': target_model.up_final.parameters(), 'lr': custom_head_lr},
    {'params': target_model.segmentation_head.parameters(), 'lr': custom_head_lr},
    {'params': target_model.adapter_conv.parameters(), 'lr': custom_head_lr},
    {'params': target_model.classifier.parameters(), 'lr': custom_head_lr}
], weight_decay=1e-4)

# Accelerated Cosine Wave Pattern with High-Velocity Floor Realignment
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, 
    T_0=5,                 
    T_mult=1,        
    eta_min=5e-5           # Reset minimum floor step size to protect against parameter freezing
)

# TRACKING METRICS SET TO CLASSIFICATION ACCURACY MILESTONES
best_val_acc = 0.0         
best_model_weights = copy.deepcopy(target_model.state_dict())

print("\n🚀 Launching Stabilized High-Velocity Dual-GPU Adversarial Loop...")
print("=" * 90)

for epoch in range(epochs):

    # ============================================================
    # TRAINING PASS
    # ============================================================
    model.train()

    running_total = 0.0
    running_class = 0.0
    running_dice = 0.0
    running_robust = 0.0

    correct_train = 0
    total_train = 0
    batches_processed = 0

    # 🚀 CHANGED: Explicit Epoch 1 Linear Learning Rate Warmup to stabilize Batch Normalization layers
    if epoch == 0:
        for param_group in optimizer.param_groups:
            param_group['lr'] = param_group['lr'] * 0.1

    for batch in train_loader:
        raw_images = batch["image"].to(device)
        raw_masks = batch["mask"].to(device)
        labels = batch["label"].to(device)

        images = torch.nan_to_num(raw_images, nan=0.0, posinf=0.0, neginf=0.0)
        masks = torch.nan_to_num(raw_masks, nan=0.0, posinf=0.0, neginf=0.0)

        optimizer.zero_grad()

        clean_outputs = model(images)

        # Adversarial Mask Generation
        with torch.no_grad():
            heatmap = torch.sigmoid(clean_outputs["attention_logits"] * 2.2)
            inverse_mask = 1.0 - (heatmap * 0.65)
            noise = 0.03 * torch.randn_like(inverse_mask)
            inverse_mask = torch.clamp(inverse_mask + noise, 0.05, 0.95)

        masked_images = images * inverse_mask
        masked_outputs = model(masked_images)
        targets = {"label": labels, "mask": masks}

        total_loss, l_class, l_dice, l_robust, alpha_t, beta_t = criterion(
            clean_outputs, masked_outputs, targets, epoch=epoch
        )

        if torch.isnan(total_loss):
            continue

        total_loss.backward()
        
        # Rigid gradient clipping to catch exploding backpropagation steps across dual cards
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
        optimizer.step()

        running_total += total_loss.item()
        running_class += l_class.item()
        running_dice += l_dice.item()
        running_robust += l_robust.item()

        probs = torch.sigmoid(clean_outputs["class_logits"].squeeze(-1))
        preds = (probs >= 0.5).float()
        correct_train += (preds == labels).sum().item()
        total_train += labels.size(0)
        batches_processed += 1

    # Restore true targeted learning rates immediately after completing the Epoch 1 warmup phase
    if epoch == 0:
        optimizer.param_groups[0]['lr'] = encoder_lr
        for i in range(1, len(optimizer.param_groups)):
            optimizer.param_groups[i]['lr'] = custom_head_lr

    # ============================================================
    # VALIDATION PASS
    # ============================================================
    model.eval()

    val_loss = 0.0
    val_running_class = 0.0
    val_running_dice = 0.0
    val_running_robust = 0.0
    
    correct_val = 0
    total_val = 0
    val_batches_processed = 0

    with torch.no_grad():
        for batch in val_loader:
            raw_val_images = batch["image"].to(device)
            raw_val_masks = batch["mask"].to(device)
            labels = batch["label"].to(device)

            images = torch.nan_to_num(raw_val_images, nan=0.0, posinf=0.0, neginf=0.0)
            masks = torch.nan_to_num(raw_val_masks, nan=0.0, posinf=0.0, neginf=0.0)

            clean_outputs = model(images)

            heatmap = torch.sigmoid(clean_outputs["attention_logits"] * 2.2)
            inverse_mask = torch.clamp(1.0 - (heatmap * 0.65), 0.05, 0.95)

            masked_images = images * inverse_mask
            masked_outputs = model(masked_images)
            targets = {"label": labels, "mask": masks}

            v_loss, v_class, v_dice, v_robust, _, _ = criterion(
                clean_outputs, masked_outputs, targets, epoch=epoch
            )

            val_loss += v_loss.item()
            val_running_class += v_class.item()
            val_running_dice += v_dice.item()
            val_running_robust += v_robust.item()

            probs = torch.sigmoid(clean_outputs["class_logits"].squeeze(-1))
            preds = (probs >= 0.5).float()
            correct_val += (preds == labels).sum().item()
            total_val += labels.size(0)
            val_batches_processed += 1

    # ============================================================
    # METRIC AGGREGATION & REPORTING
    # ============================================================
    epoch_train_loss = running_total / max(batches_processed, 1)
    epoch_train_acc = (correct_train / max(total_train, 1)) * 100

    epoch_val_loss = val_loss / max(val_batches_processed, 1)
    epoch_val_acc = (correct_val / max(total_val, 1)) * 100

    scheduler.step()

    avg_class = running_class / max(batches_processed, 1)
    avg_dice = running_dice / max(batches_processed, 1)
    avg_robust = running_robust / max(batches_processed, 1)

    print(f"📈 Epoch [{epoch+1:02d}/{epochs:02d}]")
    print(f"    [TRAIN] Loss: {epoch_train_loss:.4f} | Acc: {epoch_train_acc:.2f}%")
    print(f"    [VAL]   Loss: {epoch_val_loss:.4f} | Acc: {epoch_val_acc:.2f}%")
    print(f"    [COMP]  Avg L_class: {avg_class:.4f} | Avg L_Dice: {avg_dice:.4f} | Avg L_robust: {avg_robust:.4f}")
    print(f"    [SCHED] Alpha_t: {alpha_t:.3f} | Beta_t: {beta_t:.3f} | Head LR: {optimizer.param_groups[-1]['lr']:.7f}")
    print("-" * 90)

    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        best_model_weights = copy.deepcopy(target_model.state_dict())

SAFE_SAVE_PATH = "BraTS2020_Robust_XAI_Model.pth"
torch.save(best_model_weights, SAFE_SAVE_PATH)

print(f"\n💾 Optimization Loop Complete! Best Model Accuracy achieved: {best_val_acc:.2f}%")
print(f"Saved at: {SAFE_SAVE_PATH}")

🖥️ Total Available GPUs Detected: 2
🖥️ Primary Execution Target Device: cuda
📡 High-Performance Activation: Wrapping architecture over 2 discrete GPUs!

🚀 Launching Stabilized High-Velocity Dual-GPU Adversarial Loop...
📈 Epoch [01/50]
    [TRAIN] Loss: 0.1674 | Acc: 61.91%
    [VAL]   Loss: 2.0506 | Acc: 59.89%
    [COMP]  Avg L_class: 0.1674 | Avg L_Dice: 0.9510 | Avg L_robust: 0.0111
    [SCHED] Alpha_t: 0.000 | Beta_t: 0.000 | Head LR: 0.0004570
------------------------------------------------------------------------------------------
